***PREPROCESSING — STEP 1: Caricamento dataset***

In [1]:
# Import librerie
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# Carico il CSV in un DataFrame
df = pd.read_csv("C:/Users/matteo.segatto/Desktop/TicketClassifier/data/TicketEstrazione090326.csv")

***PREPROCESSING — STEP 2: Normalizzazione campo area***

In [2]:
# Mapping completo chiave → valore dei campi Area numerici, basato sulla tabella zhc_dropdown_maia
AREA_MAPPING = {
    '1': 'Amministrazione',
    '2': 'Amm-Economato',
    '3': 'Utenti e Ospiti',
    '4': 'CSS',
    '5': 'Personale',
    '6': 'Hardware',
    '7': 'CCE',
    '8': 'CBA DR STP',
}

# Quanti ticket hanno area numerica?
area_numerica_mask = df['area'].astype(str).str.strip().isin(AREA_MAPPING.keys())
print(f"Ticket con area numerica da mappare: {area_numerica_mask.sum():,}")
print(df[area_numerica_mask]['area'].value_counts())

# Applico mapping
df['area'] = df['area'].apply(
    lambda x: AREA_MAPPING.get(str(x).strip(), x) if pd.notna(x) else x
)

print(f"\nDistribuzione area dopo mapping:")
print(df['area'].value_counts(dropna=False))

Ticket con area numerica da mappare: 853
area
6    842
1      4
5      3
4      3
3      1
Name: count, dtype: int64

Distribuzione area dopo mapping:
area
area_personale                     14263
ciclo_passivo                      10993
ciclo_attivo                        9236
area_sanitaria                      8748
NaN                                 5849
rendicontazione_flussi              2859
protocollo_documentale_delibere     2581
area_sistemistica                   1109
sistema381                          1006
Hardware                             842
area_territoriale                    537
protocollo_delibere                  228
business_intelligence                149
Amministrazione                        4
Personale                              3
CSS                                    3
Utenti e Ospiti                        1
flussi_regionali                       1
Name: count, dtype: int64


***PREPROCESSING — STEP 3: Imputazione priorita_iniziale_cliente***

In [3]:
# Se trovo NaN in priorità iniziale = operatore NON ha modificato la priorità
# Il cliente aveva già scelto quella giusta
# priorita_iniziale_cliente == priorita_finale
# Imputo i NaN con priorita_finale

nan_prima = df['priorita_iniziale_cliente'].isna().sum()
print(f"NaN prima dell'imputazione: {nan_prima:,} ({nan_prima/len(df)*100:.1f}%)")

df['priorita_iniziale_cliente'] = df['priorita_iniziale_cliente'].fillna(df['priorita_finale'])

nan_dopo = df['priorita_iniziale_cliente'].isna().sum()
print(f"NaN dopo l'imputazione: {nan_dopo:,}")

# Distribuzione dopo imputazione
print(f"\nDistribuzione priorita_iniziale_cliente dopo imputazione:")
print(df['priorita_iniziale_cliente'].value_counts().sort_index())

NaN prima dell'imputazione: 51,401 (88.0%)
NaN dopo l'imputazione: 0

Distribuzione priorita_iniziale_cliente dopo imputazione:
priorita_iniziale_cliente
P1    12872
P2    14933
P3    29415
P4     1192
Name: count, dtype: int64


***PREPROCESSING — STEP 4: Pulizia testi***

In [ ]:
# import librerie per pulizia testo
from bs4 import BeautifulSoup
import re

# ─────────────────────────────────────────────────────────────────────────────
# PATTERN DISCLAIMER ZHC
#
# Molti ticket (soprattutto area_sistemistica) contengono in coda la firma
# completa dell'operatore ZHC con indirizzo fisico e il disclaimer GDPR:
#
#   Zucchetti Healthcare Srl       ← inizio firma con indirizzo
#   V.le Trento, 56 | 38068 Rovereto (TN)
#   Il contenuto di questa e-mail e degli eventuali allegati è strettamente
#   confidenziale ...               ← disclaimer GDPR
#   ... ufficio.privacy@zucchetti.it ← fine disclaimer
#
# Questi testi non contengono informazioni utili sulla natura del ticket
# e creano artefatti nel modello (i termini del disclaimer diventano
# falsamente discriminanti per area_sistemistica).
#
# Strategia: tagliamo a partire dal primo anchor riconoscibile.
#   1. "Zucchetti Healthcare Srl"  → rimuove firma + indirizzo + disclaimer
#   2. "Il contenuto di questa e-mail" → fallback solo per il disclaimer GDPR
# ─────────────────────────────────────────────────────────────────────────────

# Anchor 1: nome azienda nella firma — appare quasi esclusivamente nel footer
PATTERN_FIRMA_ZHC = re.compile(
    r'\s*Zucchetti Healthcare Srl.*',
    flags=re.IGNORECASE | re.DOTALL
)

# Anchor 2: inizio del testo del disclaimer GDPR (fallback)
PATTERN_DISCLAIMER_ZHC = re.compile(
    r'\s*Il contenuto di questa e-mail.*',
    flags=re.IGNORECASE | re.DOTALL
)


def rimuovi_disclaimer_zhc(testo: str) -> str:
    """
    Rimuove la firma e il disclaimer GDPR di Zucchetti Healthcare dal testo.

    Prova prima l'anchor più specifico (firma completa con indirizzo),
    poi il solo disclaimer GDPR come fallback.
    Restituisce il testo pulito con spazi extra rimossi.
    """
    if not isinstance(testo, str):
        return testo

    # Prova 1: taglia dalla firma completa (cattura anche indirizzo)
    testo = PATTERN_FIRMA_ZHC.sub('', testo)

    # Prova 2: rimuovi il disclaimer GDPR se ancora presente
    testo = PATTERN_DISCLAIMER_ZHC.sub('', testo)

    return testo.strip()


# ─────────────────────────────────────────────────────────────────────────────
# FUNZIONI DI PULIZIA HTML
# ─────────────────────────────────────────────────────────────────────────────

def pulisci_html(testo):
    """Pulizia base HTML usando BeautifulSoup."""
    # se il valore non è stringa o è vuoto/non significativo restituisco stringa vuota
    if not isinstance(testo, str) or not testo.strip():
        return ""
    # costruisco un DOM con BeautifulSoup
    soup = BeautifulSoup(testo, "html.parser")
    # ogni tag <br> viene sostituito con un newline
    for br in soup.find_all("br"):
        br.replace_with("\n")
    # per ogni <p> inserisco uno spazio dopo il tag e lo "sciolgo"
    # (unwrap rimuove il tag lasciando il contenuto)
    for p in soup.find_all("p"):
        p.insert_after(" ")
        p.unwrap()
    # estraggo il testo finale e tolgo spazi iniziali/finali
    return soup.get_text().strip()


def pulisci_blocco_reperibilita(soup):
    """
    Rimuove il blocco 'Orario reperibilità + Telefono' 
    navigando il DOM.
    """
    # prendo tutti i <br> perchè da lì riconosco la struttura del blocco
    br_tags = soup.find_all("br")
    testo = soup.get_text()
    
    # se nel testo compare la stringa di reperibilità (con o senza accento)
    if "Orario reperibilità" in testo or "Orario reperibilita" in testo:
        # verifico che ci siano almeno due <br> (atteso formattazione del blocco)
        if len(br_tags) >= 2:
            second_br = br_tags[1]
            # estraggo tutti i nodi precedenti al secondo <br> (cioè l'intero blocco)
            for elem in list(second_br.previous_siblings):
                elem.extract()
            # estraggo anche il secondo <br> stesso
            second_br.extract()
    return soup


def pulisci_descrizione(testo):
    """
    Pulizia descrizione ticket:
    - Rimuove blocco reperibilità/telefono
    - Rimuove HTML
    - Rimuove firma e disclaimer GDPR ZHC
    """
    # guard clause: input non valido → stringa vuota
    if not isinstance(testo, str) or not testo.strip():
        return ""
    
    # parse HTML
    soup = BeautifulSoup(testo, "html.parser")
    # elimino l'eventuale blocco di reperibilità
    soup = pulisci_blocco_reperibilita(soup)
    
    # normalizzo i tag <br> e <p> come in pulisci_html
    for br in soup.find_all("br"):
        br.replace_with("\n")
    for p in soup.find_all("p"):
        p.insert_after(" ")
        p.unwrap()
    
    testo_pulito = soup.get_text()
    # tolgo triple (o più) newline consecutivi -> lascio al massimo due
    testo_pulito = re.sub(r'\n{3,}', '\n\n', testo_pulito)

    # rimuovo firma e disclaimer ZHC (artefatto che inquina il testo)
    testo_pulito = rimuovi_disclaimer_zhc(testo_pulito)

    return testo_pulito.strip()


def pulisci_conversazione(testo):
    """
    Pulizia conversazione completa:
    - Divide per separatore ---
    - Pulisce ogni messaggio singolarmente
    - Rimuove blocco reperibilità solo dal primo messaggio cliente
    - Ricostruisce la conversazione
    """
    # guard clause
    if not isinstance(testo, str) or not testo.strip():
        return ""
    
    # i messaggi sono separati dal marcatore esplicito
    messaggi = testo.split('\n---\n')
    messaggi_puliti = []
    
    # itero su tutti i messaggi
    for i, messaggio in enumerate(messaggi):
        # cerco di separare l'autore dal corpo (formato 'AUTORE: testo')
        if ': ' in messaggio:
            autore, corpo = messaggio.split(': ', 1)
        else:
            autore, corpo = '', messaggio
        
        soup = BeautifulSoup(corpo, "html.parser")
        
        # solo se l'autore è CLIENTE rimuovo il blocco reperibilità
        if autore.upper() == 'CLIENTE':
            soup = pulisci_blocco_reperibilita(soup)
        
        # sostituisco <br> e <p> come prima
        for br in soup.find_all("br"):
            br.replace_with("\n")
        for p in soup.find_all("p"):
            p.insert_after(" ")
            p.unwrap()
        
        corpo_pulito = soup.get_text().strip()
        
        # se il messaggio è vuoto dopo la pulizia lo salto
        if not corpo_pulito:
            continue
        
        # ricostruisco la stringa del messaggio con autore se presente
        if autore:
            if autore.upper() == 'CLIENTE':
                autore_norm = 'CLIENTE'
            else:
                autore_norm = 'OPERATORE'
            messaggi_puliti.append(f"{autore_norm}: {corpo_pulito}")
        else:
            messaggi_puliti.append(corpo_pulito)
    
    # riunisco la conversazione con lo stesso separatore iniziale
    return '\n---\n'.join(messaggi_puliti)

# Applica le funzioni
df['oggetto_pulito']        = df['oggetto'].apply(pulisci_html)
df['descrizione_pulita']    = df['descrizione'].apply(pulisci_descrizione)
df['conversazione_pulita']  = df['conversazione'].apply(pulisci_conversazione)

In [5]:
# Analisi lunghezza descrizione
df['desc_n_char'] = df['descrizione_pulita'].str.len()
df['desc_n_parole'] = df['descrizione_pulita'].str.split().str.len()

print("=== STATISTICHE LUNGHEZZA DESCRIZIONE ===")
print(df[['desc_n_char', 'desc_n_parole']].describe().round(1))
print(f"\nDescrizioni vuote o quasi (< 5 parole): {(df['desc_n_parole'] < 5).sum():,}")
print(f"Descrizioni molto corte (< 10 parole):   {(df['desc_n_parole'] < 10).sum():,}")

=== STATISTICHE LUNGHEZZA DESCRIZIONE ===
       desc_n_char  desc_n_parole
count      58412.0        58412.0
mean         477.6           68.2
std          984.2          131.9
min            0.0            0.0
25%          193.0           27.0
50%          297.0           43.0
75%          464.0           68.0
max        65533.0         8673.0

Descrizioni vuote o quasi (< 5 parole): 504
Descrizioni molto corte (< 10 parole):   1,989


***Test***

***Test rimozione disclaimer ZHC***

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TEST RIMOZIONE DISCLAIMER ZHC
#
# Verifichiamo tre cose:
#   1. Il disclaimer viene rimosso correttamente nei ticket che ce l'hanno
#   2. I ticket senza disclaimer non vengono modificati (test di regressione)
#   3. Nessun ticket diventa vuoto per effetto della rimozione
# ─────────────────────────────────────────────────────────────────────────────

ANCHOR_DISCLAIMER = 'il contenuto di questa e-mail'

# Quante descrizioni ancora contengono il disclaimer dopo la pulizia?
n_residui = df['descrizione_pulita'].str.lower().str.contains(ANCHOR_DISCLAIMER, na=False).sum()
print(f'Ticket con disclaimer ancora presente dopo pulizia: {n_residui}')
print(f'(atteso: 0)\n')

# Quante descrizioni sono diventate vuote dopo la pulizia?
n_vuote = (df['descrizione_pulita'].str.strip() == '').sum()
print(f'Descrizioni vuote dopo pulizia: {n_vuote}')
print(f'(dovrebbe essere uguale o vicino al valore precedente alla modifica)\n')

# Mostra un esempio prima/dopo per verifica visiva
# Cerchiamo un ticket che aveva il disclaimer nel grezzo ma non nella descrizione pulita
mask_aveva_disclaimer = df['descrizione'].str.contains('Zucchetti Healthcare Srl', case=False, na=False)
print(f'Ticket con firma ZHC nella descrizione grezza: {mask_aveva_disclaimer.sum():,}')

if mask_aveva_disclaimer.sum() > 0:
    idx_esempio = df[mask_aveva_disclaimer].index[0]
    print(f'\n--- DESCRIZIONE GREZZA (ultimi 400 caratteri) ---')
    print(df.loc[idx_esempio, 'descrizione'][-400:])
    print(f'\n--- DESCRIZIONE PULITA (ultimi 400 caratteri) ---')
    print(df.loc[idx_esempio, 'descrizione_pulita'][-400:])

In [6]:
mask_rep = df['conversazione'].str.contains('reperibilit', case=False, na=False)
print(f"Ticket con blocco reperibilità nella conversazione grezza: {mask_rep.sum():,}")

if mask_rep.sum() > 0:
    idx = df[mask_rep].index[4]
    print("\n=== CONVERSAZIONE GREZZA (primi 6000 chars) ===")
    print(df.loc[idx, 'conversazione'][:6000])
    print("\n=== CONVERSAZIONE PULITA (primi 6000 chars) ===")
    print(df.loc[idx, 'conversazione_pulita'][:6000])

Ticket con blocco reperibilità nella conversazione grezza: 26,998

=== CONVERSAZIONE GREZZA (primi 6000 chars) ===
CLIENTE: <strong>Orario reperibilità contatto: </strong><br /><strong>Telefono: </strong><br /><div class="ql-editor"><p>CIAO IVAN,</p><p><br /></p><p>Stiamo cambiando connettività nelle varie strutture adesso presso la Casa di Riposo Villa del Pensionato - Via San Francesco 3 - 47017 Rocca San Casciano (FC) l'indirizzo IP pubblico è il seguente: 79.62.249.228</p><p><br /></p><p> </p><p><br /></p><p>Si segnala l'urgenza</p><p><br /></p><p> </p><p><br /></p><p>Cordiali Saluti</p></div>
---
Mauro Zaltron: <p>Effettuata modifica IP</p><br />
<p>attendo riscontro</p>
---
CLIENTE: <div class="ql-editor"><p>LA STRUTTURA MI HA COMUNICATO CHE FUNZIONANO I PROGRAMMI 1.0 CHE SI UTILLIZZANO TRAMITE DESKTOP REMOTO</p></div>
---
Mauro Zaltron: <div class="mozaik-inner" style="font-family:Arial, Helvetica, sans-serif;font-size:14px;line-height:22.4px;color:#444444;background-color:rgba(

In [7]:
# Visualizzo l'url del ticket per verifica manuale
print(f"\nURL del ticket: {df.loc[idx, 'url_ticket']}")


URL del ticket: https://maia.zucchettihc.it/index.php?module=Cases&action=DetailView&record=10160380-f462-cbd5-287a-67584f55e30f


***PREPROCESSING — STEP 6: Costruzione testo_input (oggetto_pulito + descrizione_pulita)***

In [8]:

# STEP 6: Costruzione testo_input
# Concatenazione oggetto_pulito + spazio + descrizione_pulita
# fillna('') per gestire i NaN, strip() per rimuovere spazi iniziali/finali
df['testo_input'] = (df['oggetto_pulito'].fillna('') + ' ' + df['descrizione_pulita'].fillna('')).str.strip()

# Statistiche
testi_vuoti = (df['testo_input'] == '').sum()
lunghezza_media = df['testo_input'].str.split().str.len().mean()

print(f"Testi vuoti (stringa vuota dopo strip): {testi_vuoti:,}")
print(f"Lunghezza media in parole:              {lunghezza_media:.1f}")

print(f"\nEsempio testo_input primi 600 caratteri):")
esempio = df[df['testo_input'] != '']['testo_input'].iloc[3]
print(esempio[:600])

# visualizzo url del ticket per verifica manuale
print(f"\nURL del ticket: {df.loc[df['testo_input'] != '', 'url_ticket'].iloc[3]}")


Testi vuoti (stringa vuota dopo strip): 0
Lunghezza media in parole:              73.4

Esempio testo_input primi 600 caratteri):
Bilancio di previsione Buongiorno,
sto cercando di accedere a "Bilancio di previsione", ma mi compare l'errore in allegato.
Come devo procedere?
Grazie
Laura

URL del ticket: https://maia.zucchettihc.it/index.php?module=Cases&action=DetailView&record=100255c2-d55c-bb9c-8e2e-691da5650cc5


***PREPROCESSING — STEP 7: Features derivate (n_parole, has_urgenza)***

In [9]:
# Contiamo quante parole ha ogni testo_input
df['n_parole'] = df['testo_input'].str.split().str.len()

# --- has_urgenza ---
# True se contiene almeno una keyword di urgenza forte
URGENZA_KW = ['urgente', 'immediato', 'il prima possibile', 'critico']

def calc_has_urgenza(testo):
    if not isinstance(testo, str):
        return False
    testo_lower = testo.lower()
    return any(kw in testo_lower for kw in URGENZA_KW)

# Applica la funzione
df['has_urgenza']      = df['testo_input'].apply(calc_has_urgenza)

# Verifico distribuzione e correlazione con priorità
for col in ['has_urgenza']:
    n_true = df[col].sum()
    pct = n_true / len(df) * 100
    print(f"\n{col}: {n_true:,} ticket ({pct:.1f}%)")
    print(pd.crosstab(df[col], df['priorita_finale'], normalize='index').mul(100).round(1))


has_urgenza: 3,251 ticket (5.6%)
priorita_finale    P1    P2    P3   P4
has_urgenza                           
False            12.2  28.9  56.3  2.7
True             43.5  33.3  22.9  0.3


***PREPROCESSING — STEP 8: Rimozione ticket senza informazione utile***

In [10]:
# Mostriamo la distribuzione prima di filtrare
print("=== Distribuzione lunghezze testo_input ===")
bins   = [0, 3, 5, 10, 20, 50, float('inf')]
labels = ['<=3', '3-5', '5-10', '10-20', '20-50', '50+']
print(pd.cut(df['n_parole'], bins=bins, labels=labels).value_counts().sort_index())

# Scegliamo di rimuovere i ticket con meno di 3 parole
# Sotto questa soglia non c'è abbastanza testo per capire il problema
mask_troppo_corti = df['n_parole'] < 3

print(f"\nTicket da rimuovere (< 3 parole): {mask_troppo_corti.sum():,}")
print(f"Ticket rimanenti:                  {(~mask_troppo_corti).sum():,}")

# Mostriamo qualche esempio di quello che stiamo rimuovendo
print("\n=== Esempi ticket rimossi ===")
print(df[mask_troppo_corti][['testo_input', 'priorita_finale']].head(10).to_string())

# Rimozione
df = df[~mask_troppo_corti].copy()

=== Distribuzione lunghezze testo_input ===
n_parole
<=3         43
3-5        132
5-10       819
10-20     4354
20-50    25975
50+      27089
Name: count, dtype: int64

Ticket da rimuovere (< 3 parole): 27
Ticket rimanenti:                  58,385

=== Esempi ticket rimossi ===
               testo_input priorita_finale
691       PARTENZA CLIENTE              P3
4815    aggiornamento 2025              P3
6145       VERIFICA UTENTI              P3
6207   fattura elettornica              P3
12913     FORMAZIONE SVAMA              P3
13457        aggiornamento              P3
13686  aggiornamento 26.01              P3
14805        ERRORE PAGOPA              P3
15291  aggiornamento 26.01              P3
19108  VERIFICHE VERIFICHE              P3


In [11]:
df.columns

Index(['url_ticket', 'case_number', 'oggetto', 'descrizione',
       'priorita_finale', 'priorita_iniziale_cliente', 'area',
       'tipologia_intervento', 'articolo', 'modulo_sw', 'reparto',
       'data_creazione', 'data_risoluzione', 'conversazione', 'oggetto_pulito',
       'descrizione_pulita', 'conversazione_pulita', 'desc_n_char',
       'desc_n_parole', 'testo_input', 'n_parole', 'has_urgenza'],
      dtype='str')

***PREPROCESSING — STEP 9b: Colonne booleane keyword per AreaClassifier***

Feature binarie ricavate da keyword statisticamente discriminanti per le classi
con F1 basso (analisi TF-IDF + chi-quadro su `KeywordAnalysis_Area.ipynb`).

Ogni colonna vale `True` se il termine compare in `testo_input`, `False` altrimenti.
Le colonne vengono poi usate come feature aggiuntive nell'AreaClassifier
(concatenate alla matrice embedding prima del fit LinearSVC).

In [ ]:
# Lavoriamo su testo in minuscolo per il matching (non modifichiamo testo_input)
testo = df['testo_input'].str.lower().fillna('')

# ── sistema381 ────────────────────────────────────────────────────────────────
# Il modello confonde sistema381 con area_personale per ticket generici su dati/anagrafiche.
# Questi termini identificano quasi esclusivamente l'applicativo sistema381
# (precision 0.97-1.0 dal chi2).
df['kw_s381_codice'] = (
    testo.str.contains(r'\b381\b', regex=True) |     # "381" come parola intera
    testo.str.contains('sistema 381', regex=False) |
    testo.str.contains('sistema381',  regex=False)
)
df['kw_s381_rapportino'] = (
    testo.str.contains('rapportino', regex=False) |
    testo.str.contains('rapportini', regex=False)
)
df['kw_s381_timbrate']           = testo.str.contains('timbrate',          regex=False)
df['kw_s381_calendario_presenze'] = testo.str.contains('calendario presenze', regex=False)

# ── area_territoriale ─────────────────────────────────────────────────────────
# Classe molto piccola (536 ticket). "UnoDomo" è il nome del software
# per servizi domiciliari — presenza quasi esclusiva in area_territoriale.
df['kw_ter_unodomo'] = (
    testo.str.contains('unodomo',   regex=False) |
    testo.str.contains('uno domo',  regex=False) |
    testo.str.contains(r'\bdomo\b', regex=True)   # "domo" come parola intera
)
df['kw_ter_distretto'] = testo.str.contains('distretto', regex=False)

# ── ciclo_passivo ─────────────────────────────────────────────────────────────
# Terminologia di contabilità fornitori. Precision 0.78-1.0 dal chi2.
df['kw_pas_iva']        = testo.str.contains('iva',        regex=False)
df['kw_pas_fornitore']  = (
    testo.str.contains('fornitore',  regex=False) |
    testo.str.contains('fornitori',  regex=False)
)
df['kw_pas_cespiti']    = (
    testo.str.contains('cespiti',    regex=False) |
    testo.str.contains('cespite',    regex=False)
)
df['kw_pas_prima_nota']   = testo.str.contains('prima nota',   regex=False)
df['kw_pas_ammortamento'] = testo.str.contains('ammortamento', regex=False)
df['kw_pas_analitica']    = testo.str.contains('analitica',    regex=False)
df['kw_pas_reverse']      = testo.str.contains('reverse',      regex=False)

# ── ciclo_attivo ──────────────────────────────────────────────────────────────
# "fattura/fatture" escluse (precision 0.51-0.55, troppo ambigue).
# I termini rimasti sono specifici di fatturazione attiva e rette strutture.
df['kw_att_retta']    = (
    testo.str.contains('retta',  regex=False) |
    testo.str.contains('rette',  regex=False)
)
df['kw_att_pagopa']             = testo.str.contains('pagopa',             regex=False)
df['kw_att_sdd']                = testo.str.contains('sdd',                regex=False)
df['kw_att_portale_utenti']     = testo.str.contains('portale utenti',     regex=False)
df['kw_att_fattura_elettronica'] = testo.str.contains('fattura elettronica', regex=False)

# ── Verifica distribuzione ────────────────────────────────────────────────────
KW_COLS = [c for c in df.columns if c.startswith('kw_')]
print(f'Colonne keyword aggiunte: {len(KW_COLS)}')
print()
print(f'{"Colonna":<35} {"True":>8} {"True%":>7}')
print('-' * 55)
for col in KW_COLS:
    n = df[col].sum()
    print(f'{col:<35} {n:>8,} {n/len(df)*100:>6.1f}%')

***Salvataggio Dataframe Pulito***

In [ ]:
# STEP 9 — Selezione colonne finali e salvataggio dataset_clean.csv

KW_COLS = [c for c in df.columns if c.startswith('kw_')]

COLONNE_FINALI = [
    'url_ticket',
    'case_number',
    'testo_input',
    'priorita_finale',
    'priorita_iniziale_cliente',
    'area',
    'articolo',
    'modulo_sw',
    'has_urgenza',
    'n_parole',
    'data_creazione',
] + KW_COLS

df_clean = df[COLONNE_FINALI].copy()

print(f"Righe: {len(df_clean):,}")
print(f"Colonne: {len(df_clean.columns)}  ({len(KW_COLS)} keyword)")
print(f"\nNaN per colonna (non-keyword):")
print(df_clean[COLONNE_FINALI[:11]].isnull().sum())
print(f"\nDistribuzione priorita_finale:")
print(df_clean['priorita_finale'].value_counts().sort_index())

df_clean.to_csv("../data/dataset_clean.csv", index=False, encoding='utf-8-sig')
print("\nSalvato in: data/dataset_clean.csv ✅")